In [0]:
%sql
CREATE OR REPLACE TABLE data_warehouse_factory.gold.fct_daily_employee_utilization AS
WITH 
-- 1. Zakres dat produkcyjnych (tylko dni robocze z kalendarza)
working_days AS (
  SELECT 
    d.date_key,
    d.date AS production_date
  FROM data_warehouse_factory.gold.dim_date d
  WHERE d.is_working_day = TRUE
    AND d.date BETWEEN (SELECT MIN(start_date) FROM data_warehouse_factory.gold.fct_final_operator_assignments)
                   AND (SELECT MAX(start_date) FROM data_warehouse_factory.gold.fct_final_operator_assignments)
),

-- 2. Aktywni pracownicy pobrani wprost z wymiaru Gold
active_employees AS (
  SELECT 
    employee_key, 
    employee_full_name
  FROM data_warehouse_factory.gold.dim_employees
  WHERE is_active = TRUE
),

-- 3. Siatka dostępności: Pracownik x Dzień roboczy (Norma 8.0h)
employee_days_capacity AS (
  SELECT 
    d.date_key,
    d.production_date,
    e.employee_key,
    e.employee_full_name,
    8.0 AS total_capacity_hours
  FROM working_days d
  CROSS JOIN active_employees e
),

-- 4. Przepracowane godziny z zaplanowanej produkcji (generujemy date_key z start_date)
planned_hours_worked AS (
  SELECT 
    CAST(date_format(start_date, 'yyyyMMdd') AS INT) AS date_key,
    assigned_operator_key AS employee_key,
    ROUND(SUM(timestampdiff(MINUTE, interval_start, interval_end)) / 60.0, 2) AS planned_production_hours
  FROM data_warehouse_factory.gold.fct_final_operator_assignments
  WHERE assigned_operator_key IS NOT NULL
  GROUP BY CAST(date_format(start_date, 'yyyyMMdd') AS INT), assigned_operator_key
),

-- 5. Dodatkowe godziny ze zdarzeń (z faktu Gold połączonego ze słownikiem dim_events)
additional_activities_hours AS (
  SELECT 
    e.date_key,
    e.employee_key,
    ROUND(SUM(e.duration_minutes) / 60.0, 2) AS additional_activity_hours
  FROM data_warehouse_factory.gold.fct_events_daily e
  INNER JOIN data_warehouse_factory.gold.dim_events d
    ON e.event_type = d.event_type_name
  WHERE d.is_utilized_time = TRUE
  GROUP BY e.date_key, e.employee_key
),

-- 6. Złożenie bilansu i wyliczenie czasu niewykorzystanego (Unutilized)
calculated_utilization AS (
  SELECT 
    c.date_key,
    c.production_date,
    c.employee_key,
    c.employee_full_name,
    c.total_capacity_hours,
    
    COALESCE(p.planned_production_hours, 0.0) AS planned_production_hours,
    COALESCE(a.additional_activity_hours, 0.0) AS additional_activity_hours,
    
    -- Łączny efektywny czas (maksymalnie 8h normy dziennej)
    LEAST(
      c.total_capacity_hours, 
      COALESCE(p.planned_production_hours, 0.0) + COALESCE(a.additional_activity_hours, 0.0)
    ) AS total_utilized_hours,
    
    -- Czas niezagospodarowany (8h - wykorzystany czas)
    GREATEST(
      0.0, 
      c.total_capacity_hours - (COALESCE(p.planned_production_hours, 0.0) + COALESCE(a.additional_activity_hours, 0.0))
    ) AS unutilized_hours
  FROM employee_days_capacity c
  LEFT JOIN planned_hours_worked p 
    ON c.date_key = p.date_key AND c.employee_key = p.employee_key
  LEFT JOIN additional_activities_hours a 
    ON c.date_key = a.date_key AND c.employee_key = a.employee_key
)

SELECT 
  md5(concat_ws('||', cast(date_key as string), cast(employee_key as string))) AS utilization_key,
  date_key,
  production_date,
  employee_key,
  employee_full_name,
  total_capacity_hours,
  planned_production_hours,
  additional_activity_hours,
  total_utilized_hours,
  unutilized_hours,
  
  ROUND((total_utilized_hours / total_capacity_hours) * 100, 2) AS utilization_pct,
  CURRENT_TIMESTAMP() AS _gold_created_at
FROM calculated_utilization;